# PENGWIN: pedidos del profesor (gamma por bloque, EMA en segmentación, HD95/ASSD, cuantización)
Ejecutar las celdas en orden en un entorno con GPU L4. Las celdas 5 y 6 corren mientras se entrena; la 7 y la 8, cuando la celda de espera diga que terminó.

In [ ]:
!nvidia-smi
import torch, os; print(torch.__version__, torch.cuda.is_available(), os.cpu_count(), 'CPU')

In [ ]:
# 1) Código del repo (embebido; el repo de GitHub es privado)
import base64, io, pathlib, tarfile
CODIGO = pathlib.Path("/content/codigo"); CODIGO.mkdir(exist_ok=True)
tarfile.open(fileobj=io.BytesIO(base64.b64decode("..."))).extractall(CODIGO)
print("código listo | commit 2f40fa2")

In [ ]:
# 2) Credenciales de Kaggle SIN escribirlas en el notebook.
# Colab: ícono de la llave -> secreto KAGGLE_API_TOKEN (activar "acceso desde notebook").
import os, getpass
try:
    from google.colab import userdata
    os.environ["KAGGLE_API_TOKEN"] = userdata.get("KAGGLE_API_TOKEN")
except Exception:
    os.environ["KAGGLE_API_TOKEN"] = getpass.getpass("Token de Kaggle: ")
!pip -q install -U kaggle SimpleITK
!kaggle kernels list --mine --page-size 1 | head -3   # si falla aquí, el token no está bien puesto

In [ ]:
# 3) Datos: caché 384 (train, val y test) + índice, desde la salida del kernel de datos
!mkdir -p /content/datos
import subprocess, time
for intento in range(4):
    r = subprocess.run('kaggle kernels output gustaveloper/pengwin-s10-datos -p /content/datos --file-pattern "(cache_384/.*)|(index.csv)|(resumen.json)" -q', shell=True)
    if r.returncode == 0: break
    print("reintento", intento + 1); time.sleep(10)
!ls /content/datos | head; du -sh /content/datos

In [ ]:
# 4) Checkpoint de H6: subir outputs/s10/h6_auggeo_rot35_canon_ema_w32_e60_s42.pth desde el repo (botón de archivos o este diálogo)
import pathlib, shutil
SALIDA = pathlib.Path("/content/salida"); SALIDA.mkdir(exist_ok=True)
H6 = SALIDA / "h6_auggeo_rot35_canon_ema_w32_e60_s42.pth"
if not H6.exists():
    from google.colab import files
    for nombre in files.upload():
        shutil.move(nombre, H6)
print(H6, H6.stat().st_size // 1024, "KB")

In [ ]:
# 5) Lanzar los dos entrenamientos en segundo plano (comparten la L4, como en la semana 10)
import subprocess
CORRIDAS = {
    "g1_cbam3g_auggeo_rot35_canon_ema_w32_e60_s42": "--size 384 --epochs 60 --base-channels 32 --secondary-weight 3 --augment --cosine --batch-size 16 --per-epoch 2500 --max-minutes 170 --eval-every 10 --seed 42 --aug-geo 1.0 --aug-rot-deg 35 --canonical --cbam 3 --learnable-gamma --ema 0.999",
    "e1_emaseg_auggeo_rot35_canon_w32_e60_s42": "--size 384 --epochs 60 --base-channels 32 --secondary-weight 3 --augment --cosine --batch-size 16 --per-epoch 2500 --max-minutes 170 --eval-every 10 --seed 42 --aug-geo 1.0 --aug-rot-deg 35 --canonical --cbam 1 --ema 0.999 --ema-scope segmentacion"
}
procesos = {}
for nombre, args in CORRIDAS.items():
    log = open(SALIDA / f"{nombre}.log", "w")
    cmd = f"python src/train_s10.py --cache-root /content/datos {args} --output {SALIDA}/{nombre}.pth"
    procesos[nombre] = subprocess.Popen(cmd, shell=True, cwd=CODIGO, stdout=log, stderr=subprocess.STDOUT)
    print("lanzado", nombre)

In [ ]:
# 6) Mientras entrenan: HD95 y ASSD de TODOS los fragmentos de H6 (mismas opciones que el reporte)
# Test: H6 ya está congelado; esto solo agrega métricas al reporte, no elige nada.
for split in ("val", "test"):
    cmd = (f"cd {CODIGO} && python src/infer_volume.py --checkpoint {H6} --cache-dir /content/datos/cache_384 "
           f"--split {split} --tta --distance-nofill --surface-metrics --output {SALIDA}/superficie_h6_{split}.json")
    !{cmd} 2>&1 | tail -25

In [ ]:
# Espera: correr esta celda hasta que diga que los dos terminaron (unos 70-90 min en total)
import time
for nombre, p in procesos.items():
    while p.poll() is None:
        print(nombre, "entrenando...", open(SALIDA / f"{nombre}.log").read().strip().splitlines()[-1][:120])
        time.sleep(300)
    print(nombre, "terminó con código", p.returncode)
!grep -h "γ final" /content/salida/*.log

In [ ]:
# 7) Evaluar g1 y e1 en el volumen completo de validación (TTA), igual que H6
for nombre in CORRIDAS:
    cmd = (f"cd {CODIGO} && python src/evaluate_volumes.py --checkpoint {SALIDA}/{nombre}.pth "
           f"--cache-dir /content/datos/cache_384 --split val --tta --output {SALIDA}/{nombre}_val.json")
    !{cmd} 2>&1 | tail -15
import json
g = json.loads((SALIDA / "g1_cbam3g_auggeo_rot35_canon_ema_w32_e60_s42.json").read_text())
print("gamma final por bloque:", g["gamma_final"])

In [ ]:
# 8) Cuantización de H6 en CPU (sin entrenamientos corriendo). Se elige con validación.
cmd = (f"cd {CODIGO} && python src/cuantizar.py --checkpoint {H6} --cache-dir /content/datos/cache_384 "
       f"--splits data/splits.json --split val --tta --threads {os.cpu_count()} "
       f"--output {SALIDA}/cuantizacion_h6_val.json")
!{cmd} 2>&1 | grep -v -i warn | tail -12

In [ ]:
# 9) Subir resultados a un dataset privado de Kaggle (para bajarlos luego al repo)
import json
(SALIDA / "dataset-metadata.json").write_text(json.dumps({"title": "pengwin-profesor", "id": "gustaveloper/pengwin-profesor", "licenses": [{"name": "CC0-1.0"}]}))
!kaggle datasets version -p /content/salida -m "pedidos del profesor" --dir-mode zip 2>&1 | tail -2 || true
!kaggle datasets create -p /content/salida --dir-mode zip 2>&1 | tail -2